# Eksperimen RQ1 + RQ2 (3 model) - vLLM

Notebook lengkap: **install + token + clone + run + push**, semua di sini. **Run All**, ketik token saat diminta.

- GPU besar (A6000 48 GB) -> Qwen2.5-7B presisi penuh, otomatis.
- RQ1 & RQ2 lintas 3 model: Qwen2.5-7B, Llama-3.1-8B, Gemma-2-9B (tiap model proses sendiri -> tak OOM).
- Config GPU dipilih otomatis di kode (`experiments/_kaggle_env.py`).

> **Token diketik lewat prompt (getpass), TIDAK disimpan di file** -> aman walau notebook ini di repo publik.
> Jangan pernah hardcode token di sini.

## 1. Install library

In [ ]:
!pip install -q vllm numpy huggingface_hub

## 2. Token (diketik saat run, tidak tersimpan)
Cell ini **minta ketik token** lewat prompt (getpass) - tidak nempel di file .ipynb.
- **HF_TOKEN**: download Llama & Gemma (gated). Terima lisensi dulu di halaman HF model.
- **GITHUB_TOKEN**: push hasil (opsional). Tekan Enter kosong = skip push.

Sudah `export` di terminal? Cell ini otomatis pakai itu, tak nanya lagi.

In [ ]:
import os, getpass
if not os.environ.get('HF_TOKEN'):
    tok = getpass.getpass('HF_TOKEN (Enter kosong = skip Llama/Gemma): ')
    if tok:
        os.environ['HF_TOKEN'] = tok
if not os.environ.get('GITHUB_TOKEN'):
    tok = getpass.getpass('GITHUB_TOKEN (Enter kosong = skip push): ')
    if tok:
        os.environ['GITHUB_TOKEN'] = tok
print('HF_TOKEN diset    :', 'HF_TOKEN' in os.environ)
print('GITHUB_TOKEN diset:', 'GITHUB_TOKEN' in os.environ)

## 3. Ambil repo
Pakai repo yang ada kalau notebook dibuka di dalamnya (tarik update); kalau berdiri sendiri, clone fresh ke `~/creditaudit`.

In [ ]:
import os, subprocess, torch
from pathlib import Path

def _find_repo():
    p = Path.cwd()
    for cand in [p, *p.parents]:
        if (cand / 'src' / 'gearts' / 'harness.py').exists():
            return cand
    return None

root = _find_repo()
if root is None:
    root = Path.home() / 'creditaudit'
    if not root.exists():
        subprocess.run(['git', 'clone',
                        'https://github.com/cliprompter/creditaudit.git', str(root)], check=True)
    else:
        subprocess.run(['git', '-C', str(root), 'pull'], check=False)
else:
    subprocess.run(['git', '-C', str(root), 'pull'], check=False)

os.chdir(root)
print('REPO_ROOT =', root, '| GPU:', torch.cuda.device_count())
assert os.path.isfile('src/gearts/harness.py'), 'repo tidak lengkap'
data = 'data/processed/benchmark_acuan.jsonl'
assert os.path.isfile(data), f'{data} tidak ada'
print('benchmark OK:', sum(1 for _ in open(data, encoding='utf-8')), 'baris')

## 4. DIAGNOSTIK - output MENTAH model (4 sampel)
Cek model ikut format `LANGKAH`/`JAWABAN` dan label benar.

In [ ]:
!python experiments/debug_dump.py 4

## 5. RQ1 - 3 model
Qwen2.5-7B, Llama-3.1-8B, Gemma-2-9B; tiap model proses terpisah. Tanpa HF_TOKEN, Llama/Gemma dilewati (Qwen tetap jalan).

In [ ]:
!python experiments/run_rq1_multi.py

## 6. RQ2 - 3 model (kurva grounding vs token)
Tiap model diuji di 4 panjang penalaran (panjang -> pendek -> cap-2 -> cap-1). Roster sama dengan RQ1.

In [ ]:
!python experiments/run_rq2_multi.py

## 7. Lihat hasil

In [ ]:
for p in ('experiments/rq1/tabel_rq1.md', 'experiments/rq2/kurva_rq2.md'):
    print('=' * 8, p, '=' * 8)
    print(open(p, encoding='utf-8').read())

## 8. Push hasil ke repo (butuh GITHUB_TOKEN)
Kalau GITHUB_TOKEN kosong -> skip; unduh manual `experiments/rq1` & `rq2`. Setelah push, di laptop jalankan `git pull`.

In [ ]:
import os
if os.environ.get('GITHUB_TOKEN'):
    get_ipython().system('bash scripts/push_results.sh')
else:
    print('GITHUB_TOKEN kosong -> skip push. Unduh manual experiments/rq1 & experiments/rq2.')